# 📊 Stage 5 — Analytics & Final Report

**Goal:** Compute dataset-level statistics and produce the final clean output.

This notebook covers:
- Numeric statistics (prep time, servings)
- Missing rate analysis
- Top-N ingredient frequency
- Tag distribution
- Visualisations: histograms, bar charts, pie chart
- Saving `clean_recipes.json` + `pipeline_summary.json`

---


## 0. Setup

In [ ]:
import sys, json, os
from datetime import datetime

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from collections import Counter

from src.analytics import analyse_dataset, export_pipeline_summary
from src.validation import validate_all

INPUT_FILE    = os.path.join(REPO_ROOT, 'data', 'imputed_data.json')
SCHEMA_FILE   = os.path.join(REPO_ROOT, 'recipe_schema.json')
REPORT_FILE   = os.path.join(REPO_ROOT, 'data', 'validation_report.csv')
OUT_CLEAN     = os.path.join(REPO_ROOT, 'clean_recipes.json')
OUT_SUMMARY   = os.path.join(REPO_ROOT, 'data', 'pipeline_summary.json')

plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
start_ts = datetime.now()
print('Setup complete.')


In [ ]:
with open(INPUT_FILE,  'r', encoding='utf-8') as f:
    recipes = json.load(f)
with open(SCHEMA_FILE, 'r', encoding='utf-8') as f:
    schema  = json.load(f)

report_rows = pd.read_csv(REPORT_FILE).to_dict('records') if os.path.getsize(REPORT_FILE) > 0 else []
print(f'Loaded {len(recipes)} imputed recipes.')


## 1. Dataset-Level Statistics

In [ ]:
stats = analyse_dataset(recipes)

overview = {
    'Total recipes':          stats['total_recipes'],
    'Duplicate recipes':      stats['duplicate_count'],
    'Recipes w/ imputed time': stats['imputed_prep_time_count'],
    'Recipes w/ imputed svc': stats['imputed_servings_count'],
    'Missing time rate':      f"{stats['missing_rate_prep_time_pct']}%",
    'Missing servings rate':  f"{stats['missing_rate_servings_pct']}%",
    'Avg ingredients/recipe': stats['avg_ingredients_per_recipe'],
    'Avg steps/recipe':       stats['avg_steps_per_recipe'],
}
pd.DataFrame(overview.items(), columns=['Metric', 'Value'])


In [ ]:
print('── Prep Time (minutes) ────────────────────')
pd.DataFrame([stats['prep_time_minutes_stats']])


In [ ]:
print('── Servings ────────────────────────────────')
pd.DataFrame([stats['servings_stats']])


## 2. Prep Time Distribution

In [ ]:
times   = [r['prep_time_minutes'] for r in recipes if r.get('prep_time_minutes') is not None]
was_imp = ['prep_time_minutes' in r.get('_meta', {}).get('imputed_fields', []) for r in recipes]

real_times = [t for t, imp in zip(
    [r['prep_time_minutes'] for r in recipes],
    was_imp
) if not imp]
imp_times = [t for t, imp in zip(
    [r['prep_time_minutes'] for r in recipes],
    was_imp
) if imp]

fig, ax = plt.subplots(figsize=(9, 4))
bins = range(0, max(times) + 20, 10)
ax.hist(real_times, bins=bins, color='#2ecc71', alpha=0.8, label='Known')
ax.hist(imp_times,  bins=bins, color='#e74c3c', alpha=0.8, label='Imputed (median)')
ax.axvline(stats['prep_time_minutes_stats']['median'], color='navy',
           linestyle='--', linewidth=2, label=f"Median = {stats['prep_time_minutes_stats']['median']:.0f} min")
ax.set_xlabel('Prep Time (minutes)', fontsize=12)
ax.set_ylabel('Number of Recipes', fontsize=12)
ax.set_title('Prep Time Distribution (Post-Imputation)', fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(REPO_ROOT, 'data', 'prep_time_final.png'), dpi=120)
plt.show()


## 3. Servings Distribution

In [ ]:
servings_vals = [r['servings'] for r in recipes if r.get('servings') is not None]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(servings_vals, bins=12, color='#9b59b6', edgecolor='white', alpha=0.85)
ax.axvline(stats['servings_stats']['median'], color='red',
           linestyle='--', linewidth=2, label=f"Median = {stats['servings_stats']['median']:.0f}")
ax.set_xlabel('Servings', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Servings Distribution', fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(REPO_ROOT, 'data', 'servings_dist.png'), dpi=120)
plt.show()


## 4. Top-15 Ingredients by Frequency

In [ ]:
top_ing = stats['top_15_ingredients']
names_i = [e['item'] for e in top_ing]
counts  = [e['count'] for e in top_ing]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(names_i[::-1], counts[::-1],
               color=plt.cm.viridis(np.linspace(0.2, 0.8, len(names_i))),
               edgecolor='white')
ax.bar_label(bars, padding=3, fontsize=9)
ax.set_xlabel('Frequency (across all recipes)', fontsize=12)
ax.set_title('Top-15 Most Common Ingredients', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(REPO_ROOT, 'data', 'top_ingredients.png'), dpi=120)
plt.show()


## 5. Tag Distribution

In [ ]:
tag_dist   = stats['tag_distribution']
tag_labels = list(tag_dist.keys())
tag_vals   = list(tag_dist.values())
colors     = ['#1abc9c','#3498db','#9b59b6','#e74c3c','#f39c12','#2ecc71','#e67e22']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Pie
ax1.pie(tag_vals, labels=tag_labels, autopct='%1.0f%%',
        colors=colors[:len(tag_labels)], startangle=140,
        textprops={'fontsize': 10})
ax1.set_title('Tag Distribution (pie)', fontsize=12)

# Bar
ax2.bar(tag_labels, tag_vals, color=colors[:len(tag_labels)], edgecolor='white')
ax2.set_ylabel('Count')
ax2.set_title('Tag Distribution (bar)', fontsize=12)
ax2.tick_params(axis='x', rotation=30)
for bar, val in zip(ax2.patches, tag_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             str(val), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(REPO_ROOT, 'data', 'tag_distribution.png'), dpi=120)
plt.show()


## 6. Validation Issue Breakdown

In [ ]:
if report_rows:
    rdf = pd.DataFrame(report_rows)
    print('Validation issues by severity:')
    print(rdf['severity'].value_counts().to_string())
    print('\nBy check_type:')
    print(rdf['check_type'].value_counts().to_string())
else:
    print('No validation issues in report.')


## 7. Final Schema Re-validation (Post-Imputation)

In [ ]:
final_report = validate_all(recipes, schema)
final_errors = [r for r in final_report if r['severity'] == 'error']
print(f'Post-imputation errors: {len(final_errors)}')
if final_errors:
    pd.DataFrame(final_errors)
else:
    print('✅ All recipes pass schema validation after imputation.')


## 8. Save Final Outputs

In [ ]:
# Clean recipes (remove _meta for end-consumer if desired; keeping here for auditability)
with open(OUT_CLEAN, 'w', encoding='utf-8') as f:
    json.dump(recipes, f, indent=2, ensure_ascii=False)
print(f'✅ Clean dataset  → {OUT_CLEAN}')

# Pipeline summary
export_pipeline_summary(recipes, report_rows, stats, start_ts, path=OUT_SUMMARY)
print(f'✅ Pipeline summary → {OUT_SUMMARY}')

print('\n🎉 Pipeline complete!')
print('='*55)
print(f"  Total recipes    : {stats['total_recipes']}")
print(f"  Duplicates       : {stats['duplicate_count']}")
print(f"  Recipes imputed  : {stats['imputed_servings_count']}")
print(f"  Validation issues: {len(report_rows)}")
print('='*55)


## 9. Generated Files Summary

| File | Description |
|------|-------------|
| `data/parsed_data.json` | Raw text → structured dicts |
| `data/enriched_data.json` | + SI units, tags, duplicate flags |
| `data/imputed_data.json` | + filled missing values |
| `data/validation_report.csv` | All validation issues |
| `data/imputation_log.json` | Audit log of imputed values |
| `clean_recipes.json` | **Final clean dataset** |
| `data/pipeline_summary.json` | Machine-readable run summary |
| `data/*.png` | Visualisation images |
